# 06 · Reproducción desde el Model Registry (kernel limpio)

Este notebook no entrena nada ni importa los notebooks anteriores. Carga el modelo `champion` desde el registro, los mismos datos (el snapshot, verificado por su hash) y la misma división, y comprueba que la métrica de prueba **coincide exactamente** con la registrada.

In [1]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")   # avisos de esquema de MLflow
import os, logging
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT", "1")
logging.getLogger("mlflow").setLevel(logging.ERROR)   # avisos repetidos de serialización/entorno
import numpy as np
import mlflow
from mlflow import MlflowClient

import src.pipeline as P
import src.tracking as T
from src.datos import cargar_obt

T.configurar()
cliente = MlflowClient()
version = cliente.get_model_version_by_alias(T.MODELO_REGISTRADO, "champion")
run = cliente.get_run(version.run_id)
print(f"models:/{T.MODELO_REGISTRADO}@champion → versión {version.version} · run_id {version.run_id}")
print(f"commit del código: {run.data.tags['git_commit']}")

models:/ventas-hora-matriz-modelo@champion → versión 1 · run_id b2a93515cf00415f9b32336eadbe00bf
commit del código: c542319bea14550e2933810a3b36fed35a804da3


In [2]:
snap = cargar_obt()
assert snap.hash == run.data.params["datos_hash"], "el snapshot no es el mismo que entrenó el modelo"
print(f"Datos verificados: {snap.tabla_origen} · corte {snap.fecha_corte} · hash {snap.hash[:16]}…")
test = P.dividir(P.agregar_features(snap.datos)).test

Datos verificados: PSET2.OBT.OBT_SALES_HOURLY · corte 2026-11-07 · hash d40705c0073c3866…


In [3]:
modelo = mlflow.pyfunc.load_model(f"models:/{T.MODELO_REGISTRADO}@champion")
pred = modelo.predict(test[P.ENTRADA])
umbral = float(version.tags["umbral_decision"])
reproducido = {
    "test_wape": P.wape(test[P.TARGET], pred),
    "test_mae": P.mae(test[P.TARGET], pred),
    "test_costo_negocio": P.costo_negocio(test[P.TARGET], pred, test.staff_present, umbral=umbral)["costo"],
}
for k, v in reproducido.items():
    registrado = run.data.metrics[k]
    print(f"{k:<20} registrado {registrado:.10f} · reproducido {v:.10f} · {'OK' if np.isclose(v, registrado, rtol=0, atol=1e-9) else 'DIFERENTE'}")
    assert np.isclose(v, registrado, rtol=0, atol=1e-9)
print("\nLa métrica registrada se reproduce exactamente desde models:/…@champion.")

test_wape            registrado 0.3635456015 · reproducido 0.3635456015 · OK
test_mae             registrado 2.3565078199 · reproducido 2.3565078199 · OK
test_costo_negocio   registrado 83.2600000000 · reproducido 83.2600000000 · OK

La métrica registrada se reproduce exactamente desde models:/…@champion.
